# [방법론 1 · 2026-10-26] 반사실 성과 분해 — 배포 정책(β=30 30분)과 나머지 β에 적용

`crypto/experiments/attribution.py`(커밋 `4f3f508`)를 그대로 불러와, 각 β 정책의 보고 수익(원본 평가)을 층별로 분해한다.

| 층 | 반사실 비교 |
|---|---|
| 시작 상태 효과 | 원본 평가(첫날 등가중 100%) − 2주 워밍업 평가 |
| 정적 노출 | 44종목 등가중 바스켓을 정책의 평균 노출로 고정 보유 (비용 전) |
| 노출 타이밍 | 같은 바스켓을 정책의 노출 경로대로 보유 − 정적 노출 |
| 종목 선택 | 정책(비용 전) − 노출 경로대로 보유한 바스켓 |
| 거래 비용 | 정책 비용 전 − 비용 후 |
| (선택) 실행 제약 | 운용금 330 / 2,000 USDT, 최소 주문금액 5 USDT 실행 시뮬레이션 − 워밍업 평가 |

로그 성장률로 계산해 층의 합이 정확히 보고 수익과 같다(모듈 안에서 항등식을 assert). 워밍업 정책 경로의 가치가 기존 감사
노트북(`crypto_start_state_audit_rt1006`)의 값과 같은지도 먼저 확인한다(β=30 검증 −0.08%, 테스트 +2.86%).

- **`런타임 > 모두 실행`만 누르면 된다.** CPU로 충분, 약 30~60분. 학습 없음.
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/rt1006/crypto_attribution.csv`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 준비: 데이터·환경·평가 함수 (β=30으로 치환) — 원본 crypto_beta_robustness_colab(커밋 0e86a41) 셀 재사용
run_cells('crypto/notebooks/crypto_beta_robustness_colab.ipynb', [3, 5, 6, 8, 10, 12, 13, 15, 17], patch_after=3, extra="BETA = 30   # 재학습 후 검증 샤프 최고 β\nPOOL_DIR_ROBUST = MODEL_DIR / 'crypto_beta_robustness'\nDEPLOYED_AGENT_PATH = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'\nassert DEPLOYED_AGENT_PATH.exists(), f'재학습한 v4 β={BETA} 모델 없음: {DEPLOYED_AGENT_PATH}'\nprint(f'[β 변경] BETA={BETA} | 기준 모델: {DEPLOYED_AGENT_PATH}')\n")


In [ ]:
import sys, subprocess
REPO = Path('/content/ToBeRich')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/winbean-kwon/ToBeRich.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '4f3f508df45a1ea7287cb6cec0527f96a891e323'], check=True)
sys.path.insert(0, str(REPO))
from crypto.experiments.attribution import attribute, portfolio_log_growth
print('분해 모듈 로드 완료')


In [ ]:
%%time
WARMUP = 14 * 48
BETAS = [-90, -30, 30, 90]
POOL_V4 = MODEL_DIR / 'crypto_hrl_pool_v4'
models = {b: PPO.load(str(POOL_V4 / f'agent_beta_{b}.zip')) for b in BETAS}


def policy_path(model, buckets, start):
    """start_state_audit의 run_policy와 같은 루프 — 스텝별 보유 비중·수익률·거래 가능 마스크와 시작 비중을 기록."""
    if start == 'warmup':
        i0 = all_buckets.index(buckets[0]); lo = max(0, i0 - WARMUP)
        run_bk, skip = all_buckets[lo:i0] + list(buckets), i0 - lo
    else:
        run_bk, skip = list(buckets), 0
    env = SparseGatedCryptoPortfolioEnv(buckets=run_bk, top_k=TOP_K, **env_kwargs)
    obs, _ = env.reset()
    if start == 'warmup':
        env.weights = np.zeros(env.n_assets, dtype=np.float32); obs = env._get_obs()
    init = env.weights.copy()
    Ws, Rs, Ts, done = [], [], [], False
    while not done:
        b = env.buckets[env.t]
        a, _ = model.predict(obs, deterministic=True)
        tw, rho, top_idx = env.action_to_weights(a)
        w = (1 - ALPHA) * env.weights + ALPHA * tw
        w_held = _mask_untradable(w, env._tradable(b), renorm=False)
        if env.t == skip:
            init = env.weights.copy()   # 평가 시작 시점의 보유 비중
        Ws.append(w_held); Rs.append(env._asset_ret(b)); Ts.append(env._tradable(b))
        obs, _, done, _, _ = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    v = np.asarray(env.value_history[skip:])
    return np.asarray(Ws[skip:], np.float64), np.asarray(Rs[skip:], np.float64), np.asarray(Ts[skip:]), init, v[-1] / v[0] - 1


In [ ]:
%%time
# 최소 주문금액 실행 시뮬레이터 — 노트북 셀에 그대로 들어간다(TC, TOP_K는 노트북 준비 셀에서 정의됨).
MIN_NOTIONAL = 5.0          # 바이낸스 대부분 USDT 마켓의 NOTIONAL 하한
MARGIN = 1.05               # 라이브 루프의 NOTIONAL_MARGIN과 동일
MV = MIN_NOTIONAL * MARGIN  # 실제로 주문을 내는 최소 금액(5.25 USDT)
PER_NAME = 3 * MV           # 동적 집중에서 종목당 최소 목표 금액
HOLD_BONUS = 1.5            # 동적 집중: 이미 보유 중인 종목은 비중을 이 배수로 쳐서 순위를 매긴다(경계 종목 매매 반복 방지)


def concentrate(w, equity, held=None, per_name=PER_NAME, max_names=10):
    """동적 집중: 총 노출(합)은 유지하고, 종목당 목표가 per_name 이상 되도록 비중 상위 n개에만 다시 나눈다.
    held(보유 중 마스크)가 주어지면 보유 종목을 HOLD_BONUS배로 우대해, 순위 경계에서 통째로 사고팔기를
    반복하며 수수료만 내는 것을 막는다."""
    x = float(w.sum())
    if x <= 0:
        return w
    n = int(np.clip(np.floor(x * equity / per_name), 1, max_names))
    score = w * np.where(held, HOLD_BONUS, 1.0) if held is not None else w
    top = np.argsort(score)[-n:]
    out = np.zeros_like(w)
    out[top] = w[top]
    s = out.sum()
    return out * (x / s) if s > 0 else out


def simulate(W, R, T, capital, mode, tc=0.001, seed=0):
    """달러 단위 실행 시뮬레이션.

    W: (S, N) 정책이 원하는 비중 경로(스무딩 후), R: (S, N) 다음 30분 수익률, T: (S, N) 거래 가능 마스크.
    mode: 'ideal'(무제약) / 'threshold'(현재 라이브) / 'concentrate'(동적 집중) /
          'round_up'(차이가 최소금액의 절반 이상이면 최소금액으로 주문) / 'stochastic'(확률적 반올림).
    순서는 백테스트와 같다: 버킷 b에서 비중을 맞추고(비용 차감) → 그 버킷의 수익률을 반영.
    """
    rng = np.random.default_rng(seed)
    S, N = W.shape
    V = np.zeros(N)              # 종목별 보유 평가액(USDT)
    cash = float(capital)
    nav = [float(capital)]
    gaps, expo = [], []
    n_orders, traded = 0, 0.0
    for t in range(S):
        tr = T[t]
        E = cash + V.sum()
        w_policy = W[t] * tr     # 거래 불가 종목 몫은 현금(백테스트 step_with_weights와 동일)
        w = concentrate(w_policy, E, held=V >= MIN_NOTIONAL) if mode == 'concentrate' else w_policy
        D = E * w
        d = (D - V) * tr
        absd = np.abs(d)
        if mode == 'ideal':
            o = d.copy()
        else:
            o = np.zeros(N)
            big = absd >= MV
            o[big] = d[big]
            sell_all = big & (d < 0) & (D < MV)   # 목표가 사실상 0이면 전량 매도(라이브와 동일)
            o[sell_all] = -V[sell_all]
            small = (~big) & (absd > 1e-12)
            if mode == 'round_up':
                m = small & (absd >= 0.5 * MV)
                o[m] = np.sign(d[m]) * MV
            elif mode == 'stochastic':
                m = small & (rng.random(N) < absd / MV)
                o[m] = np.sign(d[m]) * MV
        o = np.maximum(o, -V)                     # 보유량 이상은 못 판다
        if mode != 'ideal':
            o[(o < 0) & (-o < MIN_NOTIONAL)] = 0  # 5 USDT 미만 매도는 거래소가 거부(팔 수 없는 잔량)
        sells = np.minimum(o, 0)
        cash += -sells.sum() * (1 - tc)
        V += sells
        buys = np.maximum(o, 0)
        budget = cash / (1 + tc)
        if buys.sum() > budget:
            buys *= budget / buys.sum()
            if mode != 'ideal':
                buys[buys < MV] = 0               # 줄인 결과가 최소금액 미만이면 주문 못 함
        cash -= buys.sum() * (1 + tc)
        V += buys
        done = (sells != 0) | (buys != 0)
        n_orders += int(done.sum())
        traded += float(np.abs(sells).sum() + buys.sum())
        E2 = cash + V.sum()
        gaps.append(float(np.abs(V / E2 - w_policy).sum()) if E2 > 0 else 0.0)
        expo.append(float(V.sum() / E2) if E2 > 0 else 0.0)
        V = V * (1 + R[t])
        nav.append(cash + V.sum())
    nav = np.asarray(nav)
    return nav, {
        '평균 노출': float(np.mean(expo)),
        '평균 괴리(L1)': float(np.mean(gaps)),
        '주문 수': n_orders,
        '하루 주문 수': n_orders / (S / 48),
        '누적 거래대금/운용금': traded / capital,
    }


rows = []
for split, bk in [('valid', valid_buckets), ('test', test_buckets)]:
    for beta, model in models.items():
        W_rep, R_rep, T_rep, init_rep, ret_rep = policy_path(model, bk, 'orig')
        W_w, R_w, T_w, init_w, ret_w = policy_path(model, bk, 'warmup')
        assert np.allclose(R_rep, R_w) and len(W_rep) == len(W_w)
        chk_rep, _ = portfolio_log_growth(W_rep, R_rep, TC, init_rep)
        chk_w, _ = portfolio_log_growth(W_w, R_w, TC, init_w)
        assert abs(np.expm1(chk_w) - ret_w) < 1e-6 and abs(np.expm1(chk_rep) - ret_rep) < 1e-6, (np.expm1(chk_w), ret_w, np.expm1(chk_rep), ret_rep)
        exec_g = {}
        for cap in (330, 2000):
            nav, _ = simulate(W_w, R_w, T_w, cap, 'threshold', tc=TC)
            exec_g[cap] = float(np.log(nav[-1] / nav[0]))
        df = attribute(W_rep, W_w, R_w, T_w, TC, init_reported=init_rep, init_warm=init_w, executed_log_growth=exec_g[330])
        extra = exec_g[2000] - np.log1p(ret_w)
        for _, r in df.iterrows():
            rows.append({'구간': split, 'β': beta, '층': r['층'], '로그 성장률': r['로그 성장률'], '근사 %p': r['근사 %p']})
        rows.append({'구간': split, 'β': beta, '층': '  (선택) 실행 제약 — 운용금 2,000', '로그 성장률': extra, '근사 %p': np.expm1(extra) * 100})
        rows.append({'구간': split, 'β': beta, '층': '평균 노출', '로그 성장률': np.nan, '근사 %p': df.attrs['평균 노출'] * 100})
        print(f'{split} β={beta}: 보고 {ret_rep:+.2%} / 워밍업 {ret_w:+.2%} — 경로 재계산 일치 확인')

attr = pd.DataFrame(rows)
attr.to_csv(CRYPTO_DIR / 'crypto_attribution.csv', index=False, encoding='utf-8-sig')
pd.set_option('display.width', 220)
for split in ['valid', 'test']:
    t = attr[attr['구간'] == split].pivot_table(index='층', columns='β', values='근사 %p', sort=False)
    print(f'\n===== {split} (근사 %p — 층은 로그 성장률로 정확히 더해진다)')
    print(t.round(2).to_string())
print('\n저장 완료: crypto_attribution.csv')
